# QC-C3-01 demo - JavaScript OpenQASM simulator with configurable noise

This notebook runs the **C3-01** deliverable (GitHub: `Quantum-cdac/C3-01`). The simulator is **JavaScript (Node.js)**, not Python - the problem statement sets the environment as JavaScript, so this notebook only orchestrates `node` subprocesses.

Run the cells top to bottom:

1. **Cell 1** - checks this environment (Node.js / git) and fetches the repo read-only into a `c301-demo/` folder next to this notebook. It never deletes or resets anything: an existing clone is only fast-forwarded.
2. **Cell 2** - runs the demo and the 80-test suite when Node.js is available. If Node.js is not installed here, it prints the exact commands for any machine with Node 18+ instead of failing.
3. **Cell 3** - playground: edit the QASM string (and optional noise JSON) and run it through the simulator via `run.js`.

In [ ]:
import pathlib, shutil, subprocess, sys, urllib.request, zipfile

WORK = pathlib.Path.cwd() / "c301-demo"          # folder next to this notebook
REPO = WORK / "C3-01"
REPO_URL = "https://github.com/Quantum-cdac/C3-01.git"
ZIP_URL = "https://codeload.github.com/Quantum-cdac/C3-01/zip/refs/heads/main"

def sh(cmd, cwd=None):
    print("$", " ".join(str(c) for c in cmd))
    r = subprocess.run([str(c) for c in cmd], cwd=cwd, capture_output=True, text=True, timeout=600)
    out = (r.stdout or "")[-3000:]
    if out.strip():
        print(out)
    if r.returncode != 0:
        print("STDERR:", (r.stderr or "")[-1500:])
    return r.returncode

node = shutil.which("node")
git = shutil.which("git")
print("python :", sys.version.split()[0])
if node:
    r = subprocess.run([node, "--version"], capture_output=True, text=True)
    print("node   :", r.stdout.strip(), "at", node)
else:
    print("node   : NOT FOUND in this environment")
print("git    :", git or "NOT FOUND")

WORK.mkdir(exist_ok=True)
if REPO.exists():
    if git and (REPO / ".git").exists():
        sh([git, "pull", "--ff-only"], cwd=REPO)   # update only; never resets or deletes
    else:
        print("repo folder already present, leaving it untouched:", REPO)
elif git:
    sh([git, "clone", "--depth", "1", REPO_URL, str(REPO)])
else:
    print("git not available - downloading the public repo as a zip instead")
    zpath = WORK / "c301.zip"
    urllib.request.urlretrieve(ZIP_URL, zpath)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(WORK)
    (WORK / "C3-01-main").rename(REPO)
    zpath.unlink()

print("repo ready at:", REPO)
print("files:", sorted(p.name for p in REPO.iterdir()) if REPO.exists() else "MISSING")

In [ ]:
import pathlib, shutil, subprocess

WORK = pathlib.Path.cwd() / "c301-demo"
REPO = WORK / "C3-01"
node = shutil.which("node")

if not REPO.exists():
    print("Run Cell 1 first - the repo is not fetched yet.")
elif not node:
    print("=" * 70)
    print("Node.js is NOT installed in this notebook environment.")
    print("C3-01 is a JavaScript deliverable (per the problem statement), so it")
    print("needs any machine with Node.js 18+. On a laptop/desktop:")
    print()
    print("    git clone https://github.com/Quantum-cdac/C3-01.git")
    print("    cd C3-01")
    print("    node demo.js          # ideal + noisy simulation of every example")
    print("    npm test              # 80-test suite (uses node --test, no installs)")
    print("    node run.js examples/bell.qasm 4096 42")
    print()
    print("No dependencies to install (pure Node.js standard library).")
    print("Nothing here requires Qniverse platform jobs or credits.")
    print("=" * 70)
else:
    def sh(cmd):
        print("$", " ".join(str(c) for c in cmd))
        r = subprocess.run([str(c) for c in cmd], cwd=REPO, capture_output=True, text=True, timeout=600)
        print((r.stdout or "")[-5000:])
        if r.returncode != 0:
            print("STDERR:", (r.stderr or "")[-2000:])
        return r.returncode

    print("--- DEMO (ideal + noisy, all examples) ---")
    sh([node, "demo.js"])
    print("--- TEST SUITE (80 tests) ---")
    sh([node, "--test", "test/ideal.test.js", "test/parser.test.js",
        "test/noise.test.js", "test/validation.test.js", "test/reproducibility.test.js"])

In [ ]:
import pathlib, shutil, subprocess, json

WORK = pathlib.Path.cwd() / "c301-demo"
REPO = WORK / "C3-01"
node = shutil.which("node")

# --- edit these two, then run the cell -------------------------------------
QASM = r"""
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];
h q[0];
cx q[0],q[1];
measure q -> c;
"""
NOISE = None   # e.g. {"qubits": {"0": {"readoutError": 0.05, "t1": 50e-6, "t2": 70e-6, "gateTime": 35e-9}}}
SHOTS = 4096   # used only when NOISE is set
SEED = 42
# ----------------------------------------------------------------------------

if not REPO.exists() or not node:
    print("Run Cells 1 and 2 first (need the repo and Node.js).")
else:
    case = WORK / "playground.qasm"
    case.write_text(QASM)
    cmd = [node, "run.js", str(case)]
    env = None
    if NOISE:
        npath = WORK / "playground-noise.json"
        npath.write_text(json.dumps(NOISE))
        cmd += [str(SHOTS), str(SEED)]
        import os
        env = dict(os.environ, NOISE_JSON=str(npath))
    r = subprocess.run([str(c) for c in cmd], cwd=REPO, capture_output=True, text=True, timeout=600, env=env)
    print(r.stdout[-6000:])
    if r.returncode != 0:
        print("STDERR:", (r.stderr or "")[-2000:])